# Author BSVM vs three new $c_i$ functions

Notebook này điều khiển cùng runner với CLI. Test set không được dùng để tính priority hay chọn tham số.

Các dòng: `author_original`, `robust_hybrid`, `user_formula_1`, `user_formula_2`.

## Hai công thức người dùng

$$c_i^{(1)}=R\left[\widetilde\alpha_{y_i}^{p}r_i^{\beta}\rho_i^{\gamma}\exp\left(-\frac{|1-m_i|}{\tau}\right)\right],\quad m_i=y_if(x_i).$$

$$g_i=\frac1k\sum_{x_j\in N_k^{Val}(x_i)}\mathbf 1(y_j=y_i),$$

$$c_i^{(2)}=R\left[\widetilde\alpha_{y_i}^{p}r_i^{\beta}\rho_i^{\gamma}\exp\left(-\frac{|1-y_if(x_i)|}{\tau}\right)g_i^{\delta}\right].$$

Xem `docs/AUTHOR_CI_FORMULAS.md` để biết định nghĩa đầy đủ.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ROOT

## Chỉnh tập chạy

Dùng `SEARCH='none'` để smoke test, `'fast'` để kiểm tra rộng hơn, và `'paper'` cho grid Table 3.

In [ ]:
DATASETS = 'fruitfly'
KERNELS = 'linear'
VARIANTS = 'author_original,robust_hybrid,user_formula_1,user_formula_2'
PAPER_GROUPS = 'experiment_1'
SEARCH = 'none'
MAX_ROWS = 60
OUTPUT_DIR = 'outputs/notebook_author_ci'

In [ ]:
command = [
    sys.executable,
    str(ROOT / 'examples' / 'run_author_ci_comparison.py'),
    '--datasets', DATASETS,
    '--kernels', KERNELS,
    '--variants', VARIANTS,
    '--paper-groups', PAPER_GROUPS,
    '--search', SEARCH,
    '--max-rows', str(MAX_ROWS),
    '--output-dir', OUTPUT_DIR,
    '--resume',
]
subprocess.run(command, cwd=ROOT, check=True)

## Bảng kết quả

In [ ]:
output = ROOT / OUTPUT_DIR
comparison = pd.read_csv(output / 'comparison_all.csv')
columns = [
    'dataset', 'paper_group', 'kernel', 'algorithm', 'status',
    'validation_score', 'objective_test', 'accuracy_test',
    'f1_macro_test', 'minority_f1_test', 'support_vectors', 'model_fits'
]
comparison[[column for column in columns if column in comparison.columns]]

In [ ]:
pd.read_csv(output / 'comparison_deltas_vs_original.csv')

In [ ]:
pd.read_csv(output / 'summary_by_variant.csv')

## Full paper grid

Nên chạy lệnh dưới trong terminal vì có thể mất nhiều giờ và `--resume` sẽ ghi tiếp sau mỗi tổ hợp.

```powershell
python examples/run_author_ci_comparison.py --search paper --paper-groups all --resume
```